# Low-ℓ BB — Manual Tile Inspector (g3 source)

Same workflow as `bb_map_inspect.ipynb` but reads from a `.g3.gz` file (SPT3G format).

**To switch Stokes component**: change `STOKES_KEY` (T / Q / U) in Paths, re-run from Load map down.  
**To switch frequency**: change `FREQ` (90GHz / 150GHz / 220GHz) in Paths, re-run from Load map down.

In [ ]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from spt3g import core, maps
from astropy.io import fits
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import (
    apply_spt_style,
    mask_map,
    show_map_full_field,
    show_map_thumbnail,
)

apply_spt_style()

#### Paths

In [ ]:
# Stokes selector
# Change to "Q" or "U" and re-run from Load map to inspect a different component.
STOKES_KEY = "T"

_STOKES_FIELD = {"T": 0, "Q": 1, "U": 2}
assert STOKES_KEY in _STOKES_FIELD, f"STOKES_KEY must be T / Q / U, got '{STOKES_KEY}'"

# Frequency selector
# Change to "90GHz" or "150GHz" to switch band.
FREQ = "220GHz"

_FREQ_OPTIONS = {"90GHz", "150GHz", "220GHz"}
assert FREQ in _FREQ_OPTIONS, f"FREQ must be one of {_FREQ_OPTIONS}, got '{FREQ}'"

# g3 data file
G3_FILE = "/sptgrid/user/javva/baseline_bb_coadd_match/coadd_fullauto/full/no_signflip_bundle_000.g3.gz"

# Mask directory
MASK_DIR = "/sptlocal/user/creichardt/bb2020"

# Available masks — refer to this when choosing in Load mask
MASK_FILES = {
    "mask_250_30"   : "puremask8192_0p5medwt_250mJy_30arcmin.npz",
    "mask_250_60"   : "puremask8192_0p5medwt_250mJy_60arcmin.npz",
    "mask_250_nd30" : "puremask8192_0p5medwt_250mJy_nodisk_30arcmin.npz",
    "mask_250_nd60" : "puremask8192_0p5medwt_250mJy_nodisk_60arcmin.npz",
    "mask_100_30"   : "puremask8192_0p5medwt_100mJy_30arcmin.npz",
    "mask_apod_30"  : "puremask8192_0p5medwt_30arcmin.npz",
    "mask_apod_60"  : "puremask8192_0p5medwt_60arcmin.npz",
}

# Output directory
SAVE_DIR = "/sptlocal/user/vwelke/lowl_bb_tiles_g3"
os.makedirs(SAVE_DIR, exist_ok=True)

# Display parameters
# nside=2048 → pixel size ~1.7 arcmin, so 2.0 arcmin/px is appropriate here
RESO_ARCMIN = 2.0
PATCH_DEG   = 15.0
PATCH_PIX   = int(PATCH_DEG * 60 / RESO_ARCMIN)
CMAP        = "coolwarm"

print(f"Stokes key  : {STOKES_KEY}")
print(f"Frequency   : {FREQ}")
print(f"Patch size  : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px")
print(f"Resolution  : {RESO_ARCMIN} arcmin/px")

#### Load map

In [ ]:
# Find the frame matching FREQ
target_frame = None
for frame in core.G3File(G3_FILE):
    if frame.type == core.G3FrameType.Map and frame['Id'] == FREQ:
        target_frame = frame
        break

if target_frame is None:
    raise ValueError(f"No Map frame with Id='{FREQ}' found in {G3_FILE}")

# Extract the selected Stokes component (stored as weighted: value = Stokes * weight)
stokes_weighted = np.array(target_frame[STOKES_KEY])
nside = target_frame[STOKES_KEY].nside

# Get the diagonal weight for this Stokes component
# Note: for Q and U this is the diagonal approximation of the 2x2 pol weight matrix.
# Fine for inspection; proper science requires inverting the full [QQ, QU; QU, UU] matrix.
wpol = target_frame['Wpol']
weight = {
    'T': np.array(wpol.TT),
    'Q': np.array(wpol.QQ),
    'U': np.array(wpol.UU),
}[STOKES_KEY]

# Unweight: pixels with weight > 0 are observed
good = weight > 0
stokes_arr = np.full(len(stokes_weighted), hp.UNSEEN)
stokes_arr[good] = stokes_weighted[good] / weight[good]

print(f"Loaded     : {STOKES_KEY}  {FREQ}  from  {os.path.basename(G3_FILE)}")
print(f"nside      : {nside}")
print(f"Total pix  : {len(stokes_arr):,}")
print(f"Observed   : {good.sum():,}  ({good.sum()/len(stokes_arr)*100:.2f}% of sky)")

#### Observed-pixel mask

In [ ]:
obs_mask = np.isfinite(stokes_arr) & (stokes_arr != hp.UNSEEN)

obs_pix        = np.where(obs_mask)[0]
theta_c, phi_c = hp.pix2ang(nside, obs_pix)
ra_obs  = np.degrees(phi_c)
ra_obs  = np.where(ra_obs > 180, ra_obs - 360, ra_obs)
dec_obs = 90.0 - np.degrees(theta_c)

ra_min,  ra_max  = ra_obs.min(),  ra_obs.max()
dec_min, dec_max = dec_obs.min(), dec_obs.max()

print(f"Observed pixels  : {obs_mask.sum():,}  ({obs_mask.sum()/len(stokes_arr)*100:.2f}% of sky)")
print(f"RA  range  :  {ra_min:.2f}° → {ra_max:.2f}°   span = {ra_max-ra_min:.2f}°")
print(f"Dec range  : {dec_min:.2f}° → {dec_max:.2f}°   span = {dec_max-dec_min:.2f}°")

#### Tile grid

In [ ]:
ra_centres  = np.arange(ra_min  + PATCH_DEG/2, ra_max  + PATCH_DEG/2, PATCH_DEG)
dec_centres = np.arange(dec_min + PATCH_DEG/2, dec_max + PATCH_DEG/2, PATCH_DEG)

n_ra, n_dec = len(ra_centres), len(dec_centres)
print(f"RA  centres  : {n_ra}   [{ra_centres[0]:.1f}° … {ra_centres[-1]:.1f}°]")
print(f"Dec centres  : {n_dec}   [{dec_centres[0]:.1f}° … {dec_centres[-1]:.1f}°]")
print(f"Total patches : {n_ra * n_dec}")
print()
print("RA_IDX  →  RA centre")
for i, r in enumerate(ra_centres):
    print(f"  {i}  →  {r:+.1f}°")
print()
print("DEC_IDX  →  Dec centre")
for j, d in enumerate(dec_centres):
    print(f"  {j}  →  {d:+.1f}°")

#### Unmasked patches

Inspect the raw map first — spot bright point sources, decide which mask to use below.

In [ ]:
# Choose patch
#   RA_IDX  : 0 … n_ra-1   (west → east)
#   DEC_IDX : 0 … n_dec-1  (south → north)
RA_IDX  = 0
DEC_IDX = 0

ra_c  = ra_centres[RA_IDX]
dec_c = dec_centres[DEC_IDX]
print(f"Patch ({RA_IDX}, {DEC_IDX})  →  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°")

In [ ]:
# Plot (unmasked)
rms        = float(np.std(stokes_arr[obs_mask]))
vmin, vmax = -rms, rms

show_map_thumbnail(
    stokes_arr,
    vmin=vmin, vmax=vmax,
    title=(
        f"{STOKES_KEY}  {FREQ}  unmasked  |  patch ({RA_IDX}, {DEC_IDX})  "
        f"|  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°"
    ),
    unit="Tcmb", cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")

#### Load mask

After inspecting the raw patches above, choose a mask from `MASK_FILES` and run this cell.  
The mask is at nside=8192 and will be automatically downgraded to nside=2048 to match the map.

In [ ]:
# Choose mask
# Available keys (defined in Paths):
#   mask_250_30   — 250 mJy point-source mask, 30 arcmin apodisation
#   mask_250_60   — 250 mJy point-source mask, 60 arcmin apodisation
#   mask_250_nd30 — 250 mJy, no disk, 30 arcmin
#   mask_250_nd60 — 250 mJy, no disk, 60 arcmin
#   mask_100_30   — 100 mJy point-source mask, 30 arcmin
#   mask_apod_30  — apodisation only, 30 arcmin (no source masking)
#   mask_apod_60  — apodisation only, 60 arcmin (no source masking)
ACTIVE_MASK = "mask_250_nd30"

mask_path = os.path.join(MASK_DIR, MASK_FILES[ACTIVE_MASK])
with np.load(mask_path) as d:
    apod     = d[d.files[0]].astype(float)
    mask_key = d.files[0]

nside_mask = hp.get_nside(apod)
if nside_mask != nside:
    print(f"Downgrading mask from nside={nside_mask} → nside={nside}")
    apod = hp.ud_grade(apod, nside_out=nside)

obs_and_mask = obs_mask & (apod > 0)

# Apply apodisation weight
stokes_masked                 = stokes_arr.copy()
stokes_masked[obs_and_mask]  *= apod[obs_and_mask]
stokes_masked[~obs_and_mask]  = hp.UNSEEN

print(f"Mask loaded  : {ACTIVE_MASK}")
print(f"File         : {MASK_FILES[ACTIVE_MASK]}")
print(f"key='{mask_key}'  |  nside after downgrade={hp.get_nside(apod)}")
print(f"Pixels after mask : {obs_and_mask.sum():,}  (was {obs_mask.sum():,} unmasked)")

#### Masked patches

Same `RA_IDX` / `DEC_IDX` as above — easy before/after comparison.

In [ ]:
# Choose patch
#   Same RA_IDX / DEC_IDX as above for direct before/after comparison
RA_IDX  = 0
DEC_IDX = 0

ra_c  = ra_centres[RA_IDX]
dec_c = dec_centres[DEC_IDX]
print(f"Patch ({RA_IDX}, {DEC_IDX})  →  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°")

In [ ]:
# Plot (masked)
rms        = float(np.std(stokes_masked[obs_and_mask]))
vmin, vmax = -rms, rms

show_map_thumbnail(
    stokes_masked,
    vmin=vmin, vmax=vmax,
    title=(
        f"{STOKES_KEY}  {FREQ}  masked ({ACTIVE_MASK})  |  patch ({RA_IDX}, {DEC_IDX})  "
        f"|  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°"
    ),
    unit="Tcmb", cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")

#### Power spectrum (draft)

> Draft only — no beam correction, no mode-coupling / purification.

In [ ]:
# Load Q and U for E/B decomposition
# This is independent of STOKES_KEY — E/B decomposition needs both simultaneously.
# Requires Load mask to have been run first so apod and obs_and_mask exist.

# Re-open the file to get Q and U from the same frequency frame
for frame in core.G3File(G3_FILE):
    if frame.type == core.G3FrameType.Map and frame['Id'] == FREQ:
        _frame_qu = frame
        break

wpol_qu = _frame_qu['Wpol']

def _unweight(key, frm, wp):
    raw = np.array(frm[key])
    w   = np.array(getattr(wp, {'Q': 'QQ', 'U': 'UU'}[key]))
    arr = np.zeros(len(raw))
    good = w > 0
    arr[good] = raw[good] / w[good]
    arr[~good] = 0.0   # anafast expects 0, not UNSEEN
    return arr

Q_arr = _unweight('Q', _frame_qu, wpol_qu)
U_arr = _unweight('U', _frame_qu, wpol_qu)

# Apply the same apodisation mask
Q_arr[obs_and_mask] *= apod[obs_and_mask]
Q_arr[~obs_and_mask] = 0.0
U_arr[obs_and_mask] *= apod[obs_and_mask]
U_arr[~obs_and_mask] = 0.0

T_zeros = np.zeros(len(Q_arr))

LMAX = 600   # nside=2048 → max useful ell ~4000, but keep low for draft
cls  = hp.anafast([T_zeros, Q_arr, U_arr], lmax=LMAX)
cl_BB = cls[2]
ell   = np.arange(len(cl_BB))

print(f"Computed pseudo-C_ell up to ell={LMAX}")
print(f"Note: mask mixes E into B — BB is overestimated without mode-coupling correction.")

In [ ]:
# Plot BB pseudo-D_ell
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ell[2:], ell[2:] * (ell[2:] + 1) * cl_BB[2:] / (2 * np.pi), label="BB")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell^{BB}$  [$\mu{\rm K}^2$]")
ax.set_title(
    f"BB pseudo-$C_\\ell$  |  {FREQ}  |  {os.path.basename(G3_FILE)}  |  mask: {ACTIVE_MASK}\n"
    f"(draft — no beam correction, no E→B leakage correction)"
)
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Save C_ell to FITS — uncomment when ready
# out_cl = os.path.join(SAVE_DIR, f"cl_BB_{FREQ}_{ACTIVE_MASK}.fits")
# hp.write_cl(out_cl, cls, overwrite=True)
# print(f"C_ell saved → {out_cl}")